# SIGNATE Spectral Analysis Challenge – Experiment Log

Author: Maggie Smith  
Goal: Predict wood moisture content from NIR spectra.

This notebook tracks model experiments, preprocessing methods,
feature engineering, and competition scores.

Dataset
- Train samples: 1322
- Test samples: 550
- Spectral features: 1555
- Wavenumber range: 3999–9993 cm⁻¹

## Experiment Results

### Best Score
**20.85**

### Experiments

| Experiment | Description | Score | Observation |
|---|---|---|---|
| submission.csv | Baseline PLS model | **20.85** | Best result so far |
| submission_pls.csv | PLS variant | 20.85 | Similar performance to baseline |
| submission_pls_scaled.csv | PLS with StandardScaler | 27.58 | External scaling degraded performance |
| submission_pls_2.csv | Alternate PLS configuration | 27.58 | Similar result to scaled model |
| exp03_savgol_deriv1_species_pls12 | Savitzky–Golay derivative + species feature | 32.57 | Derivative preprocessing significantly worsened predictions |

### Observations

- Raw spectra performed better than derivative preprocessing.
- External scaling (StandardScaler) degraded model performance.
- Savitzky–Golay derivative introduced instability for this dataset.
- Baseline PLS regression remains the strongest model so far.

### Current Best Pipeline

```
Raw Spectra
→ PLS Regression
```

Score: **20.85**

### Next Experiments

1. Mean-centering spectra before PLS
2. Wider search for PLS components (10–40)
3. Wavelength region selection
4. Compare with ElasticNet and LightGBM models

In [8]:
import pandas as pd
from pathlib import Path

# -----------------------------
# Project paths
# -----------------------------
PROJECT_ROOT = Path("..")  # notebook is in /notebooks
DATA_DIR = PROJECT_ROOT / "data"

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv(DATA_DIR / "train.csv", encoding="cp932")
test = pd.read_csv(DATA_DIR / "test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (1322, 1559)
Test shape: (550, 1558)


In [5]:
spectral_cols = [c for c in train.columns if c.startswith("x")]
target = "moisture"
species_col = "species"

## Experiment 4 – Savitzky-Golay + PCA + Species
Goal: Reduce spectral noise and dimensionality

In [6]:
from scipy.signal import savgol_filter
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

In [11]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from datetime import datetime
from pathlib import Path

# -----------------------------
# Experiment metadata
# -----------------------------
experiment_name = f"exp03_savgol_deriv1_species_pls12_{datetime.now().strftime('%Y%m%d')}"

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Define spectral columns
# -----------------------------
spectral_cols = [col for col in train.columns
                 if col not in ['sample number', 'species number', '樹種', '含水率']]

# -----------------------------
# Extract spectral data
# -----------------------------
X_spec = train[spectral_cols].values
X_test_spec = test[spectral_cols].values

# -----------------------------
# Savitzky-Golay filter
# -----------------------------
X_spec_sg = savgol_filter(X_spec, window_length=11, polyorder=2, deriv=1)
X_test_spec_sg = savgol_filter(X_test_spec, window_length=11, polyorder=2, deriv=1)

# -----------------------------
# Add species feature
# -----------------------------
X_final = np.hstack([X_spec_sg, train[['species number']].values])
X_test_final = np.hstack([X_test_spec_sg, test[['species number']].values])

# -----------------------------
# Scale features
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# -----------------------------
# Train model
# -----------------------------
pls = PLSRegression(n_components=12)
pls.fit(X_scaled, train['含水率'])

# -----------------------------
# Predict
# -----------------------------
X_test_scaled = scaler.transform(X_test_final)
test_preds = pls.predict(X_test_scaled).flatten()

print("Sample predictions:", test_preds[:10])

# -----------------------------
# Create submission
# -----------------------------
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

print(submission.head())

# -----------------------------
# Save submission
# -----------------------------
import os

submissions_dir = "../submissions"

# create folder if it doesn't exist
os.makedirs(submissions_dir, exist_ok=True)

output_file = f"{submissions_dir}/{experiment_name}.csv"

submission.to_csv(output_file, index=False, header=False)

print("Submission saved to:", output_file)

# -----------------------------
# Verify saved file
# -----------------------------
check = pd.read_csv(output_file, header=None)
print("Saved file preview:")
print(check.head())

Train shape: (1322, 1559)
Test shape: (550, 1558)
Sample predictions: [200.97338401 135.83355623 123.96611201 120.56961716 114.92073276
 121.93042706 113.43633241 120.26016888 114.59501764 113.58892347]
   sample number         含水率
0             95  200.973384
1             96  135.833556
2             97  123.966112
3             98  120.569617
4             99  114.920733
Submission saved to: ../submissions/exp03_savgol_deriv1_species_pls12_20260323.csv
Saved file preview:
    0           1
0  95  200.973384
1  96  135.833556
2  97  123.966112
3  98  120.569617
4  99  114.920733


In [ ]:
import pandas as pd
import numpy as np
from scipy.signal import savgol_filter
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression
from datetime import datetime
from pathlib import Path

# -----------------------------
# Experiment metadata
# -----------------------------
experiment_name = f"exp03_savgol_deriv1_species_pls12_{datetime.now().strftime('%Y%m%d')}"

# -----------------------------
# Load data
# -----------------------------
train = pd.read_csv("../data/train.csv", encoding="cp932")
test = pd.read_csv("../data/test.csv", encoding="cp932")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

# -----------------------------
# Define spectral columns
# -----------------------------
spectral_cols = [col for col in train.columns
                 if col not in ['sample number', 'species number', '樹種', '含水率']]

# -----------------------------
# Extract spectral data
# -----------------------------
X_spec = train[spectral_cols].values
X_test_spec = test[spectral_cols].values

# -----------------------------
# Savitzky-Golay filter
# -----------------------------
X_spec_sg = savgol_filter(X_spec, window_length=11, polyorder=2, deriv=1)
X_test_spec_sg = savgol_filter(X_test_spec, window_length=11, polyorder=2, deriv=1)

# -----------------------------
# Add species feature
# -----------------------------
X_final = np.hstack([X_spec_sg, train[['species number']].values])
X_test_final = np.hstack([X_test_spec_sg, test[['species number']].values])

# -----------------------------
# Scale features
# -----------------------------
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

# -----------------------------
# Train model
# -----------------------------
pls = PLSRegression(n_components=12)
pls.fit(X_scaled, train['含水率'])

# -----------------------------
# Predict
# -----------------------------
X_test_scaled = scaler.transform(X_test_final)
test_preds = pls.predict(X_test_scaled).flatten()

print("Sample predictions:", test_preds[:10])

# -----------------------------
# Create submission
# -----------------------------
submission = pd.DataFrame({
    "sample number": test["sample number"],
    "含水率": test_preds
})

print(submission.head())

# -----------------------------
# Save submission
# -----------------------------
import os

submissions_dir = "../submissions"

# create folder if it doesn't exist
os.makedirs(submissions_dir, exist_ok=True)

output_file = f"{submissions_dir}/{experiment_name}.csv"

submission.to_csv(output_file, index=False, header=False)

print("Submission saved to:", output_file)

# -----------------------------
# Verify saved file
# -----------------------------
check = pd.read_csv(output_file, header=None)
print("Saved file preview:")
print(check.head())